# DeepConvLSTM on OPPORTUNITY (Google Colab)

Runtime → Change runtime type → **T4 GPU**. Everything (data, trained models, results) is kept in Google Drive under `MyDrive/deepconvlstm`, so it survives when Colab disconnects.

Before the first run, upload `deepconvlstm.zip` to `MyDrive/`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE = '/content/drive/MyDrive/deepconvlstm'
if not os.path.exists(DRIVE):
    !unzip -q /content/drive/MyDrive/deepconvlstm.zip -d /content/drive/MyDrive/
%cd {DRIVE}
!nvidia-smi -L

## 1. Data (only the first time)
Downloads ~300 MB from UCI and writes `data/opportunity.npz` to Drive. Later sessions skip this.

In [ ]:
if not os.path.exists('data/opportunity.npz'):
    !python download_data.py --out /content/raw
    !python preprocess.py --raw /content/raw --out data/opportunity.npz
!ls -lh data

## 2. Train
Each run is saved in `runs/<name>/` on Drive: `best_model.pt` (weights + settings), metrics, confusion matrix, learning curves.

In [ ]:
!python train.py --model deepconvlstm --task gestures --name ref_s0

## 3. Reload a saved model for further experiments

In [ ]:
from evaluate import load_trained
import numpy as np, torch
from data import load_split

model, cfg, names = load_trained('runs/ref_s0')
print(cfg['model'], cfg['task'], len(names), 'classes')

test = load_split(np.load('data/opportunity.npz'), 'test', task=cfg['task'], sensors=cfg['sensors'],
                  window=cfg['window'], step=cfg['step'], include_null=not cfg['no_null'])
with torch.no_grad():
    logits = model(test.X[:5].to(next(model.parameters()).device))
print([names[i] for i in logits.argmax(1).tolist()], [names[i] for i in test.y[:5].tolist()])

In [ ]:
!python evaluate.py --run runs/ref_s0 --save-predictions

## 4. Ablations
Run one block per session (see README). Example: block A, 3 seeds.

In [ ]:
!SEEDS='0 1 2' bash ablations.sh A

In [ ]:
!python summarize.py
import pandas as pd
pd.read_csv('results_by_config.csv')